# OpenNPC — Live Dashboard

This notebook gives a live, 4-panel view while you chat with an OpenNPC personality:

1. **Neural network panel** — a stylized diagram that lights up while GPT-2 generates. It is decorative, not a render of GPT-2's real internals (124M parameters is far too many to draw). Think of it as a "thinking" indicator.
2. **History panel** — the full conversation so far.
3. **Live process panel** — a log of what actually happened for each reply: the exact prompt sent, how many tokens went in and came out, and how long generation took.
4. **Terminal output panel** — the latest exchange, styled like a terminal window.

Every exchange is also appended to a plain-text transcript at `data/session_history.txt`.

Run the cells top to bottom (or **Run → Run All Cells**), then use the text box + Send button at the bottom to chat.

## 1. Install/import dependencies

Run once. `ipywidgets` and `networkx` are the only new packages beyond what training/inference already need.

In [ ]:
import sys
!{sys.executable} -m pip install -q ipywidgets networkx matplotlib


In [ ]:
import sys
from datetime import datetime
from pathlib import Path

# Find the repo root (the folder that contains the opennpc package), so this
# notebook works whether Jupyter was started in the repo root or in dashboard/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "opennpc").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import networkx as nx
import ipywidgets as widgets
from IPython.display import display, clear_output

from opennpc.personality import load_all
from opennpc.conversation import Conversation
from opennpc.npc import NPC
from opennpc.prompt import build_prompt
from opennpc.paths import DATA_DIR

print(f"Repo root: {ROOT}")

## 2. Load model + pick a personality

In [ ]:
PERSONALITIES = {p.name: p for p in load_all()}

personality_dropdown = widgets.Dropdown(
    options=list(PERSONALITIES),
    value="Rude",
    description="Personality:",
)
display(personality_dropdown)

In [ ]:
from opennpc.model import GPT2Backend, GPT2LoRABackend

print("Loading model, this can take a moment on first run...")
# Use the fine-tuned LoRA adapter if it has been trained, otherwise base GPT-2.
try:
    model = GPT2LoRABackend()
    BACKEND_NAME = "GPT-2 + LoRA adapter"
except FileNotFoundError as e:
    print(e)
    model = GPT2Backend()
    BACKEND_NAME = "GPT-2 (base, no fine-tuning)"

personality = PERSONALITIES[personality_dropdown.value]
conversation = Conversation()
npc = NPC(personality, conversation, model, build_prompt)

DATA_DIR.mkdir(exist_ok=True)
HISTORY_FILE = DATA_DIR / "session_history.txt"

print(f"Ready. Backend: {BACKEND_NAME} on {model.device}. Personality: {personality.name}")

## 3. The stylized neural network panel

A simple layered node diagram. Nodes "light up" more while a reply is being generated, to give a live sense of activity — decorative, not a literal visualization of GPT-2's actual architecture.

In [ ]:
import random as _r

LAYER_SIZES = [6, 8, 8, 6, 4]  # purely visual — not GPT-2's real layer sizes

def build_layout():
    G = nx.DiGraph()
    pos = {}
    node_id = 0
    layer_nodes = []
    for layer_idx, size in enumerate(LAYER_SIZES):
        ids = []
        for n in range(size):
            G.add_node(node_id)
            x = layer_idx
            y = (n - size / 2)
            pos[node_id] = (x, y)
            ids.append(node_id)
            node_id += 1
        layer_nodes.append(ids)
    for a, b in zip(layer_nodes, layer_nodes[1:]):
        for i in a:
            for j in b:
                G.add_edge(i, j)
    return G, pos, layer_nodes

NN_GRAPH, NN_POS, NN_LAYERS = build_layout()

def draw_network(activity=0.2, ax=None):
    """activity: 0.0 (idle) to 1.0 (fully 'thinking') controls how many
    nodes/edges are highlighted, purely for visual feedback."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(3.2, 4))
    ax.clear()
    node_colors = []
    for n in NN_GRAPH.nodes():
        lit = _r.random() < activity
        node_colors.append("#ffb703" if lit else "#264653")
    edge_colors = []
    for e in NN_GRAPH.edges():
        lit = _r.random() < (activity * 0.4)
        edge_colors.append("#ffb703" if lit else "#33333333")
    nx.draw(
        NN_GRAPH, NN_POS, ax=ax, node_color=node_colors, edge_color=edge_colors,
        node_size=180, with_labels=False, arrows=False, width=1.2,
    )
    ax.set_title("thinking..." if activity > 0.3 else "idle", fontsize=9)
    ax.set_axis_off()
    return ax


## 4. Build the 4-panel layout

In [ ]:
out_nn = widgets.Output(layout=widgets.Layout(border="1px solid #444", width="260px", height="380px"))
out_history = widgets.Output(layout=widgets.Layout(border="1px solid #444", width="260px", height="380px", overflow_y="auto"))
out_process = widgets.Output(layout=widgets.Layout(border="1px solid #444", width="260px", height="380px", overflow_y="auto"))
out_terminal = widgets.Output(layout=widgets.Layout(border="1px solid #444", width="260px", height="380px", overflow_y="auto"))

panels = widgets.HBox([out_nn, out_history, out_process, out_terminal])

text_input = widgets.Text(placeholder="Type a message...", layout=widgets.Layout(width="70%"))
send_button = widgets.Button(description="Send", button_style="primary")
input_row = widgets.HBox([text_input, send_button])

with out_nn:
    fig, ax = plt.subplots(figsize=(3.2, 4))
    draw_network(0.1, ax=ax)
    plt.show()

with out_process:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Ready. Personality: {personality.name}")

with out_terminal:
    print("OPENNPC")
    print(f"Personality: {personality.name}")

display(panels, input_row)


## 5. Wire up the Send button

Each send: light up the network panel → build the prompt → generate the reply → log the real numbers (prompt, tokens, time) → update history and the terminal panel → append to the saved transcript.

In [ ]:
def log_process(msg):
    with out_process:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def refresh_history():
    with out_history:
        clear_output(wait=True)
        for turn in conversation.get_history():
            speaker = "You" if turn["role"] == "user" else "NPC"
            print(f"{speaker}: {turn['text']}")

def show_network(activity):
    with out_nn:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(3.2, 4))
        draw_network(activity, ax=ax)
        plt.show()

def save_to_transcript(user_text, reply):
    with open(HISTORY_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now().isoformat()}] ({personality.name}) You: {user_text}\n")
        f.write(f"[{datetime.now().isoformat()}] ({personality.name}) NPC: {reply}\n")

def on_send(_):
    user_text = text_input.value.strip()
    if not user_text:
        return
    text_input.value = ""

    show_network(1.0)
    reply = npc.respond(user_text)
    show_network(0.1)

    # These are the real values from the call that just ran, not a script.
    stats = model.last_stats
    prompt = build_prompt(personality, conversation)
    log_process("Prompt sent to the model:\n    " + prompt.replace("\n", "\n    "))
    log_process(f"{stats['prompt_tokens']} tokens in -> {stats['new_tokens']} tokens out")
    log_process(f"Generated in {stats['seconds']:.2f}s on {stats['device']} "
                f"({stats['new_tokens'] / max(stats['seconds'], 1e-6):.1f} tokens/s)")

    refresh_history()

    with out_terminal:
        clear_output(wait=True)
        print("OPENNPC")
        print(f"Personality: {personality.name}")
        print()
        print(f"You: {user_text}")
        print(f"NPC: {reply}")

    save_to_transcript(user_text, reply)
    log_process("Saved to data/session_history.txt")

send_button.on_click(on_send)
text_input.on_submit(on_send)

## Notes

- **Switching personality mid-session:** re-run the cell in section 2 after changing the dropdown — this creates a fresh `Conversation`, so it starts a new session for the new personality (it won't overwrite the other personality's saved memory file from `main.py`, since this notebook keeps its own session transcript in `data/session_history.txt`).
- **The network diagram is decorative.** GPT-2's real architecture (12 transformer blocks, 12 attention heads each, 768-dim embeddings) is too large to usefully draw node-by-node — this stylized 5-layer diagram is a stand-in "thinking" indicator, not a literal visualization.
- **Transcript file:** `data/session_history.txt` accumulates across every run of this notebook (it's appended to, never overwritten) — delete it manually if you want a clean slate.